# NB-02 — Benchmark de LLMs com visão via OpenRouter (Cap. 3)

Reproduz a tabela `tab:valores_ia` (cosseno na página 1680-069) e acrescenta **CER/WER** (micro, como na plataforma), tempo e custo. A chamada é feita por `engines.openrouter_engine.OpenRouterEngine`; as métricas, por `avaliacao.py` (NB-03). **Cada execução cobra:** `DRY_RUN=True` por padrão, teto `MAX_USD` e cache em disco por (modelo, imagem, prompt, temperature, max_tokens).

Saídas em `saidas/NB-02/`: `resultados.csv`, `tabela_cosseno.tex`, `tabela_cer_wer.tex`, `manifest.json`.

**Gabarito:** `1680-069.txt` (do autor, ao lado da imagem). As marcações são removidas antes de medir: `--texto--` (rasura) sai; `##texto##` (correção acima da linha) fica sem as marcas. Hifenização de fim de linha mantida como no arquivo.

**Modelos:** os da plataforma (`engines/openrouter_models.json`, mantido com `deploy/openrouter/suggest_models.py`), conferidos contra o catálogo atual da OpenRouter. A lista de 14 modelos do TCC está defasada e só serve de referência na seção de aceite.

In [1]:
# Parâmetros
from pathlib import Path
DRY_RUN = True            # True: não chama a API (só mostra o plano). Mude para False para gastar.
MAX_USD = 1.00            # teto de gasto NOVO nesta execução (cache não conta)
PROMPT = None             # None = prompt do script do TCC ("Only Transcribe into Brazilian Portuguese")
TEMPERATURE = 0.0
EXCLUIR = {'nvidia/nemotron-3.5-super-vl:free'}  # estourou MAX_TOKENS (finish_reason=length) em 10-11-2026; autor tirou da lista
MAX_TOKENS = 4096         # a página tem ~2000 caracteres; modelos de raciocínio gastam parte pensando
IMAGEM = Path.home() / 'text_tcc_unb/unbtex-example/figuras/1680-069.png'
GABARITO = Path.home() / 'text_tcc_unb/unbtex-example/figuras/1680-069.txt'
SAIDA = Path('saidas/NB-02')
SEED = 0  # sem aleatoriedade neste notebook; mantido por convenção

In [2]:
import sys, json, hashlib
sys.path.insert(0, 'lib')
from datetime import datetime
import pandas as pd
import benchmark_or as B
from avaliacao import agregar

RAIZ = Path.cwd().parent
print('chave OPENROUTER_API_KEY encontrada:', B.carregar_chave(RAIZ))   # só informa se existe
PROMPT = PROMPT or B.PROMPT_TCC
referencia = B.carregar_gabarito(GABARITO)
print('gabarito:', len(referencia), 'caracteres,', len(referencia.split()), 'palavras')

chave OPENROUTER_API_KEY encontrada: True
gabarito: 2394 caracteres, 381 palavras


## 1. Modelos: o que ainda existe na OpenRouter
O TCC usou modelos de 2025; vários já foram retirados. A listagem é pública e gratuita.

In [3]:
modelos = [m['id'] for m in json.loads((RAIZ / 'engines/openrouter_models.json').read_text())['models']]
modelos = [m for m in modelos if m not in EXCLUIR]
cat = B.catalogo()
disponiveis, indisponiveis = B.disponibilidade(modelos, cat)
print(len(disponiveis), 'disponíveis;', len(indisponiveis), 'indisponíveis (ficam de fora):')
for m in indisponiveis: print('  -', m)

8 disponíveis; 0 indisponíveis (ficam de fora):


## 2. Execução

In [4]:
registros, plano = B.executar(disponiveis, IMAGEM, referencia, SAIDA, dry_run=DRY_RUN, max_usd=MAX_USD,
                              prompt=PROMPT, temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
print('a chamar:', plano['chamar']); print('em cache:', plano['cache'])

a chamar: []
em cache: ['google/gemini-3.8-flash', 'google/gemini-2.5-flash', 'anthropic/claude-haiku-5.5', 'anthropic/claude-sonnet-5.5', 'openai/gpt-6-luna', 'qwen/qwen3-vl-235b-a22b-instruct', 'mistralai/mistral-small-3.2-24b-instruct', 'z-ai/glm-5.3-flash']


## 3. Resultados

In [5]:
if not registros:
    print('Sem resultados (DRY_RUN=True ou nada executado).')
else:
    df = pd.DataFrame(registros)
    cols = ['motor','cer','wer','cosseno','tempo_ms','custo_usd','erro','cache','data','provedor','finish_reason','hipotese']
    df = df[[c for c in cols if c in df.columns]]
    SAIDA.mkdir(parents=True, exist_ok=True)
    df.to_csv(SAIDA/'resultados.csv', index=False)
    resumo = pd.DataFrame(agregar(registros)).set_index('motor')   # 1 página: micro == valor da página
    display(df.drop(columns=['hipotese']).sort_values('cosseno', ascending=False))
    print('custo novo desta execução (US$):', round(sum(r['custo_usd'] or 0 for r in registros if not r['cache']), 4))
    (SAIDA/'tabela_cosseno.tex').write_text(B.tabela_tex(registros, 'cosseno', 'Valores aferidos por modelo de IA (cosseno)', 'tab:valores_ia', 'Cosseno'), encoding='utf-8')
    (SAIDA/'tabela_cer_wer.tex').write_text(B.tabela_tex(registros, 'cer', 'CER por modelo de IA na página 1680-069', 'tab:cer_ia', 'CER', crescente=True, casas=3), encoding='utf-8')

,motor,cer,wer,cosseno,tempo_ms,custo_usd,erro,cache,data,provedor,finish_reason
0,google/gemini-3.8-flash,0.007519,0.047244,0.993864,12247.550924,0.008378,None,True,2026-11-10T14:10:22,Google,stop
3,anthropic/claude-sonnet-5.5,0.008772,0.055118,0.993864,5469.534181,0.012900,None,True,2026-11-10T14:10:39,Claude Platform on AWS,stop
2,anthropic/claude-haiku-5.5,0.009607,0.057743,0.992607,8504.007996,0.001355,None,True,2026-11-10T14:10:34,Claude Platform on AWS,stop
7,z-ai/glm-5.3-flash,0.012531,0.055118,0.991991,8923.881214,0.001138,None,True,2026-11-10T14:12:20,Mistral,stop
1,google/gemini-2.5-flash,0.015038,0.060367,0.991370,3111.000205,0.001417,None,True,2026-11-10T14:10:25,Google AI Studio,stop
4,openai/gpt-6-luna,0.010860,0.062992,0.991352,19572.729341,0.000860,None,True,2026-11-10T14:10:59,OpenAI,stop
5,qwen/qwen3-vl-235b-a22b-instruct,0.016708,0.060367,0.989490,54374.381931,0.001486,None,True,2026-11-10T14:11:53,Venice,stop
6,mistralai/mistral-small-3.2-24b-instruct,0.030911,0.094488,0.982984,17921.920422,0.000237,None,True,2026-11-10T14:12:11,DeepInfra,stop


custo novo desta execução (US$): 0


## 4. Aceite
Só `gemini-2.5-flash` e `mistral-small-3.2-24b-instruct` existem nas duas listas; para eles o cosseno deve ficar dentro de ±0,02 do Cap. 3. Divergências não são erro do notebook: modelos e prompt mudam; o TCC deve dizer a data da nova chamada.

In [6]:
TCC = {'google/gemini-2.0-flash-001':0.99,'google/gemini-2.5-flash-preview-05-20':0.92,'google/gemini-2.5-pro':0.92,
       'google/gemini-2.5-flash':0.92,'anthropic/claude-3.7-sonnet':0.92,'openai/gpt-4.1-mini':0.92,
       'meta-llama/llama-4-scout':0.92,'google/gemini-2.5-flash-preview':0.91,'meta-llama/llama-4-maverick':0.91,
       'qwen/qwen2.5-vl-72b-instruct':0.90,'anthropic/claude-sonnet-4':0.83,
       'mistralai/mistral-small-3.2-24b-instruct':0.81,'openai/gpt-4.1':0.74,'openai/gpt-4o-mini':0.60}
if registros:
    linhas = [{'modelo': r['motor'], 'tcc': TCC.get(r['motor']), 'agora': round(r['cosseno'], 2), 'dif': round(r['cosseno'] - TCC[r['motor']], 2),
               'ok_±0,02': abs(r['cosseno'] - TCC[r['motor']]) <= 0.02} for r in registros if r.get('cosseno') is not None and r['motor'] in TCC]
    display(pd.DataFrame(linhas))
    print('falhas:', [(r['motor'], r['erro']) for r in registros if r['erro']])

,modelo,tcc,agora,dif,"ok_±0,02"
0,google/gemini-2.5-flash,0.92,0.99,0.07,False
1,mistralai/mistral-small-3.2-24b-instruct,0.81,0.98,0.17,False


falhas: []


In [7]:
manifest = {
    'notebook': 'NB-02', 'data': datetime.now().isoformat(timespec='seconds'), 'dry_run': DRY_RUN,
    'imagem': str(IMAGEM), 'imagem_sha256': hashlib.sha256(IMAGEM.read_bytes()).hexdigest(),
    'gabarito_sha256': hashlib.sha256(referencia.encode()).hexdigest(),
    'prompt': PROMPT, 'temperature': TEMPERATURE, 'max_tokens': MAX_TOKENS, 'max_usd': MAX_USD,
    'modelos_executados': [r['motor'] for r in registros], 'modelos_indisponiveis': indisponiveis, 'fonte_modelos': 'engines/openrouter_models.json',
    'custo_novo_usd': sum(r['custo_usd'] or 0 for r in registros if not r['cache']),
    'obs': 'A plataforma reenvia a imagem como JPEG (qualidade 92); o script original do TCC enviava PNG.',
}
SAIDA.mkdir(parents=True, exist_ok=True)
(SAIDA/'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(json.dumps(manifest, indent=2, ensure_ascii=False)[:900])

{
  "notebook": "NB-02",
  "data": "2026-11-10T14:20:27",
  "dry_run": false,
  "imagem": "~/text_tcc_unb/unbtex-example/figuras/1680-069.png",
  "imagem_sha256": "8fb2ef3ef1b9f145955295246762ba6a69f69b3d71b9be9683a393f5106c6066",
  "gabarito_sha256": "80d42f8531f986e6a639e6bd384ee4fc369238d4377a6a3fc7f47ba23fe736d0",
  "prompt": "Only Transcribe into Brazilian Portuguese",
  "temperature": 0.0,
  "max_tokens": 4096,
  "max_usd": 1.0,
  "modelos_executados": [
    "google/gemini-3.8-flash",
    "google/gemini-2.5-flash",
    "anthropic/claude-haiku-5.5",
    "anthropic/claude-sonnet-5.5",
    "openai/gpt-6-luna",
    "qwen/qwen3-vl-235b-a22b-instruct",
    "mistralai/mistral-small-3.2-24b-instruct",
    "z-ai/glm-5.3-flash"
  ],
  "modelos_indisponiveis": [],
  "fonte_modelos": "engines/openrouter_models.json",
  "custo_novo_usd": 0,
  "obs": "A plataforma reenvia a
